# 06 — V6: Scale to 500+ TCEs + Odd/Even Transit Comparison

**Goal:** Reach precision > 80% with recall = 100% on a representative 500+ TCE test set.

Three changes vs. V5:
1. **Scale:** 500 TCEs (250 confirmed + 250 FP) from Kepler DR25 — test set has a representative FP mix instead of the skewed 16-TCE V5 split.
2. **Odd/even diff as 4th input channel** — catches period-doubled EBs that the secondary-view channel (phase=0.5 fold) doesn't reach.
3. **Synthetic pretrain fix** — tested 3 configs (from-scratch, BN reset, depth-matched) and the winner is used here. Comparison from `scripts/run_v6.py`.

This notebook visualizes channels and inspects predictions. Training and the 3-way comparison live in `scripts/run_v6.py` so the notebook stays fast to re-run.

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import numpy as np
import matplotlib.pyplot as plt

from src.models.taylor_cnn import TaylorCNN

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 1. Load V6 dataset (primary + secondary + odd/even)

In [ ]:
d = torch.load('../data/kepler_tce_v6.pt', weights_only=False)
assert 'fluxes_odd_even' in d, 'Dataset is not V6 format. Run build_dataset.py first.'

phases = d['phases']
fluxes_primary = d['fluxes']
fluxes_secondary = d['fluxes_secondary']
fluxes_odd_even = d['fluxes_odd_even']
labels = d['labels']
names = d['names']
depths = d['depths_ppm']

n_conf = int(labels.sum())
n_fp = int((1 - labels).sum())
print(f'Dataset: {len(labels)} TCEs ({n_conf} confirmed, {n_fp} FP)')
print(f'Channel shapes: phases {phases.shape}, primary {fluxes_primary.shape},')
print(f'                secondary {fluxes_secondary.shape}, odd/even {fluxes_odd_even.shape}')

## 2. Four-channel visualization — planet vs. example FPs

Each row is one TCE; four columns are the four input channels to the CNN. For a real planet, only the primary (col 2) should show a dip; the secondary (col 3) and odd/even-diff (col 4) should be flat noise. For a correct-period EB, secondary lights up. For a period-doubled EB, odd/even lights up.

In [ ]:
conf_idx = (labels == 1).nonzero(as_tuple=True)[0]
fp_idx = (labels == 0).nonzero(as_tuple=True)[0]

# Pick a few examples:
#   - deepest-primary planet (clear transit)
#   - FP with deepest secondary (correct-period EB candidate)
#   - FP with largest odd/even amplitude (period-doubled EB candidate)
p_min = fluxes_primary[:, 95:105].min(dim=1).values
s_min = fluxes_secondary[:, 95:105].min(dim=1).values
oe_amp = fluxes_odd_even[:, 95:105].abs().max(dim=1).values

ex_planet = conf_idx[p_min[conf_idx].argmin()].item()
ex_fp_sec = fp_idx[s_min[fp_idx].argmin()].item()
ex_fp_oe = fp_idx[oe_amp[fp_idx].argmax()].item()

examples = [
    (ex_planet, f'Confirmed: {names[ex_planet]}'),
    (ex_fp_sec, f'FP (deepest 2nd): {names[ex_fp_sec]}'),
    (ex_fp_oe, f'FP (largest O/E): {names[ex_fp_oe]}'),
]

fig, axes = plt.subplots(len(examples), 4, figsize=(16, 3 * len(examples)), sharex=True)
channel_titles = ['Phase', 'Primary flux', 'Secondary flux', 'Odd - Even diff']
for row, (idx, title) in enumerate(examples):
    channels = [phases[idx], fluxes_primary[idx], fluxes_secondary[idx], fluxes_odd_even[idx]]
    for col, (ch, ct) in enumerate(zip(channels, channel_titles)):
        axes[row, col].plot(phases[idx], ch, '.', ms=2, color='C0')
        axes[row, col].axhline(0, color='k', lw=0.5)
        axes[row, col].axvline(0, color='r', lw=0.5, alpha=0.3)
        axes[row, col].set_title(f'{title} — {ct}' if row == 0 else ct, fontsize=9)
    axes[row, 0].set_ylabel(title, fontsize=9)
for ax in axes[-1]:
    ax.set_xlabel('phase (rad)')
plt.tight_layout(); plt.show()

## 3. Train/val/test split (stratified, seed 42)

In [ ]:
torch.manual_seed(42)
conf_perm = conf_idx[torch.randperm(len(conf_idx))]
fp_perm = fp_idx[torch.randperm(len(fp_idx))]

def split_indices(idx, train_frac=0.7, val_frac=0.15):
    n = len(idx); nt = int(n * train_frac); nv = int(n * val_frac)
    return idx[:nt], idx[nt:nt+nv], idx[nt+nv:]

conf_train, conf_val, conf_test = split_indices(conf_perm)
fp_train, fp_val, fp_test = split_indices(fp_perm)
test_idx = torch.cat([conf_test, fp_test])

def make_split(idx):
    return (phases[idx].to(device), fluxes_primary[idx].to(device),
            fluxes_secondary[idx].to(device), fluxes_odd_even[idx].to(device),
            labels[idx].to(device))

test_ph, test_p, test_s, test_oe, test_l = make_split(test_idx)
print(f'Test set: {len(test_l)} TCEs ({int(test_l.sum())} conf, {int((1-test_l).sum())} FP)')

## 4. Load winning V6 model

Run `python -m scripts.run_v6` from the project root first — it trains three configs (from-scratch, BN-reset, depth-matched) and saves the winner to `src/models/taylor_cnn_v6.pt`. This notebook just loads and evaluates.

In [ ]:
ckpt = torch.load('../src/models/taylor_cnn_v6.pt', weights_only=False)

print(f'Winning config: {ckpt["config"]}')
print(f'Metrics: {ckpt["metrics"]}')
print(f'Taylor A:       {ckpt["A"]:.5f}')

print('\nThree-way comparison (from run_v6.py):')
print(f'  {"Config":<6} {"Acc":>7} {"Prec":>7} {"Rec":>7} {"F1":>7} {"A":>7}')
for cfg, r in ckpt['all_configs'].items():
    print(f'  {cfg:<6} {r["accuracy"]:>6.1%}  {r["precision"]:>6.1%}  '
          f'{r["recall"]:>6.1%}  {r["f1"]:>6.3f}  {r["A"]:.4f}')

model = TaylorCNN(init_amplitude=0.01).to(device)
model.load_state_dict(ckpt['state_dict'])
model.eval()
print(f'\nModel loaded: {sum(p.numel() for p in model.parameters())} params')

## 5. Evaluate on test set

In [ ]:
with torch.no_grad():
    probs = model(test_ph, test_p, test_s, test_oe).squeeze(1).cpu()
preds = (probs > 0.5).float()
tl = test_l.cpu()

TP = int(((preds == 1) & (tl == 1)).sum())
TN = int(((preds == 0) & (tl == 0)).sum())
FP = int(((preds == 1) & (tl == 0)).sum())
FN = int(((preds == 0) & (tl == 1)).sum())

acc = (TP + TN) / len(tl)
prec = TP / (TP + FP) if TP + FP else 0
rec = TP / (TP + FN) if TP + FN else 0
f1 = 2 * prec * rec / (prec + rec) if prec + rec else 0

print(f'=== V6 Test Set Results ({len(tl)} TCEs) ===')
print(f'Accuracy:  {acc:.1%}')
print(f'Precision: {prec:.1%}')
print(f'Recall:    {rec:.1%}')
print(f'F1:        {f1:.3f}')
print()
print('Confusion matrix:')
print(f'                      Pred Planet  Pred Not')
print(f'  True Planet             {TP:>3}         {FN:>3}')
print(f'  True Not-Planet         {FP:>3}         {TN:>3}')
print()
print('Comparison:')
print(f'  V4: acc 75.0%, prec 66.7%, rec 100%, F1 0.800  (16 TCEs)')
print(f'  V5: acc 81.2%, prec 72.7%, rec 100%, F1 0.842  (16 TCEs)')
print(f'  V6: acc {acc:.1%}, prec {prec:.1%}, rec {rec:.1%}, F1 {f1:.3f}  ({len(tl)} TCEs)')
print()
print(f'Success criterion (prec > 80% AND rec = 100%): '
      f'{"PASS" if (prec > 0.80 and rec >= 1.0) else "FAIL"}')

## 6. Inspect false positives by channel activity

For each FP caught or missed, show which channels light up. A surviving FP with flat secondary and flat odd/even is a non-EB failure mode (stellar activity, centroid offset) — not solvable with this architecture.

In [ ]:
test_orig = test_idx.tolist()

# Split test FPs into caught vs. missed; show channel activity for each
fp_positions = ((tl == 0).nonzero(as_tuple=True)[0]).tolist()
fp_caught = [pos for pos in fp_positions if preds[pos] == 0]
fp_missed = [pos for pos in fp_positions if preds[pos] == 1]

print(f'Test FPs: {len(fp_caught)} caught, {len(fp_missed)} survived')
print()
header = f'  {"KOI":<12} {"prob":<6} {"p_dip":<9} {"s_dip":<9} {"oe_amp":<8}'
print('Caught FPs:')
print(header)
for pos in fp_caught:
    orig = test_orig[pos]
    pd = fluxes_primary[orig, 95:105].min().item()
    sd = fluxes_secondary[orig, 95:105].min().item()
    oe = fluxes_odd_even[orig, 95:105].abs().max().item()
    print(f'  {names[orig]:<12} {probs[pos].item():.2f}   {pd:+.4f}   {sd:+.4f}   {oe:.4f}')

if fp_missed:
    print('\nSurviving FPs (model predicted planet):')
    print(header)
    for pos in fp_missed:
        orig = test_orig[pos]
        pd = fluxes_primary[orig, 95:105].min().item()
        sd = fluxes_secondary[orig, 95:105].min().item()
        oe = fluxes_odd_even[orig, 95:105].abs().max().item()
        print(f'  {names[orig]:<12} {probs[pos].item():.2f}   {pd:+.4f}   {sd:+.4f}   {oe:.4f}')
else:
    print('\nNo surviving FPs.')

## 7. Visualize a caught vs. surviving FP side by side

In [ ]:
pick = []
if fp_caught:
    pick.append((fp_caught[0], 'Caught FP'))
if fp_missed:
    pick.append((fp_missed[0], 'Surviving FP'))

if pick:
    fig, axes = plt.subplots(len(pick), 3, figsize=(14, 3 * len(pick)), sharex=True,
                             squeeze=False)
    for row, (pos, title) in enumerate(pick):
        orig = test_orig[pos]
        for col, (name_ch, arr) in enumerate([
            ('Primary',   fluxes_primary[orig]),
            ('Secondary', fluxes_secondary[orig]),
            ('Odd/Even',  fluxes_odd_even[orig]),
        ]):
            axes[row, col].plot(phases[orig], arr, '.', ms=2)
            axes[row, col].axhline(0, color='k', lw=0.5)
            axes[row, col].axvline(0, color='r', lw=0.5, alpha=0.3)
            axes[row, col].set_title(f'{title}: {names[orig]} — {name_ch} (prob={probs[pos]:.2f})', fontsize=9)
        axes[row, 0].set_ylabel(title)
    for ax in axes[-1]:
        ax.set_xlabel('phase (rad)')
    plt.tight_layout(); plt.show()
else:
    print('No FPs in test set — nothing to visualize.')